In [ ]:
!pip install requests pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
import shutil
from tqdm import tqdm

In [4]:
api_url = 'https://api.jikan.moe/v4'

RAW_DIR = 'data/raw'
TEMP_DIR = 'data/raw/temp'

if not os.path.exists(RAW_DIR):
    os.makedirs(RAW_DIR)
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

In [5]:
def make_request(url, retries=3):
    for i in range(retries):
        try:
            response = requests.get(url)
            if response.status_code == 200:
                return response.json()
            
            elif response.status_code == 429:
                wait_time = (i + 1) * 2 
                print(f"Rate limit hit. Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                print(f"Error {response.status_code}: {url}")
                return None
                
        except Exception as e:
            print(f"Exception: {e}")
            time.sleep(1)
            
    return None

In [7]:
def scrape_jikan_db(endpoint='manga', start_page=1, end_page=None):
    
    current_page = start_page
    has_next_page = True
    
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR)

    while has_next_page:
        if end_page and current_page > end_page:
            break
            
        url = f"{api_url}/{endpoint}?page={current_page}"
        
        data = make_request(url)
        
        if data and 'data' in data:
            items = data['data']
            
            temp_file = os.path.join(TEMP_DIR, f"page_{current_page}.json")
            with open(temp_file, 'w', encoding='utf-8') as f:
                json.dump(items, f, indent=4, ensure_ascii=False)
            
            pagination = data.get('pagination', {})
            has_next_page = pagination.get('has_next_page', False)
            
            print(f"Da xong trang {current_page}. (Co {len(items)} muc)")
            current_page += 1
            
            time.sleep(1.05) 
            
        else:
            print(f"Khong lay duoc du lieu trang {current_page}. Dung qua trinh.")
            break

    all_data = []
    temp_files = sorted(os.listdir(TEMP_DIR), key=lambda x: int(x.split('_')[1].split('.')[0]))
    
    for filename in tqdm(temp_files, desc="Merging"):
        filepath = os.path.join(TEMP_DIR, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            page_data = json.load(f)
            all_data.extend(page_data)
            
    output_file = f'data/raw/{endpoint}_data.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, indent=4, ensure_ascii=False)
        
    shutil.rmtree(TEMP_DIR)

scrape_jikan_db('manga', start_page=1, end_page=None)

Da xong trang 1. (Co 25 muc)
Da xong trang 2. (Co 25 muc)
Da xong trang 3. (Co 25 muc)
Da xong trang 4. (Co 25 muc)
Da xong trang 5. (Co 25 muc)
Da xong trang 6. (Co 25 muc)
Da xong trang 7. (Co 25 muc)
Da xong trang 8. (Co 25 muc)
Da xong trang 9. (Co 25 muc)
Da xong trang 10. (Co 25 muc)
Da xong trang 11. (Co 25 muc)
Da xong trang 12. (Co 25 muc)
Da xong trang 13. (Co 25 muc)
Da xong trang 14. (Co 25 muc)
Da xong trang 15. (Co 25 muc)
Da xong trang 16. (Co 25 muc)
Da xong trang 17. (Co 25 muc)
Da xong trang 18. (Co 25 muc)
Da xong trang 19. (Co 25 muc)
Da xong trang 20. (Co 25 muc)
Da xong trang 21. (Co 25 muc)
Da xong trang 22. (Co 25 muc)
Da xong trang 23. (Co 25 muc)
Da xong trang 24. (Co 25 muc)
Da xong trang 25. (Co 25 muc)
Da xong trang 26. (Co 25 muc)
Da xong trang 27. (Co 25 muc)
Da xong trang 28. (Co 25 muc)
Da xong trang 29. (Co 25 muc)
Da xong trang 30. (Co 25 muc)
Da xong trang 31. (Co 25 muc)
Da xong trang 32. (Co 25 muc)
Da xong trang 33. (Co 25 muc)
Da xong trang 34. (

Merging: 100%|██████████| 3262/3262 [00:03<00:00, 851.83it/s] 


In [ ]:
def clean_and_drop_attributes(input_file, output_file):
    # Load file json
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    print(f"Bo du lieu co {df.shape[0]} dong, {df.shape[1]} cot")

    # Giu lai cac cot su dung cho RAG Chatbot va Graph

    # De tao co so du lieu cho mo hinh can mal_id, url lam khoa
    # De xay dung mo hinh RAG Chatbot can title, synopsis, background, genres,...
    # De xay dung do thi tri thuc can cac cot title, genre, type, authors
    # De tinh toan cac chi so va do dac tuong quan can cac thuoc tinh dang numeric, status, 

    keep_cols = [
        'mal_id', 'url', 'title', 'title_english', 'status', 'type', 
        'chapters', 'volumes', 'score', 'scored_by', 'rank', 
        'popularity', 'members', 'favorites', 'synopsis', 'background',
        'published', 'images', 'genres', 'themes', 'demographics', 'authors'
    ]
    
    # Chi giu lai cac cot trong dataframe
    existing_cols = [col for col in keep_cols if col in df.columns]
    df = df[existing_cols]

    # Lam phang du lieu

    # Rut gon danh sach dict thanh danh sach ten
    # [{'name': 'Action'}, {'name': 'Comedy'}] -> ['Action', 'Comedy']
    def extract_names(row_list):
        if isinstance(row_list, list):
            return [item.get('name') for item in row_list if 'name' in item]
        return []

    # Ap dung cho genres, themes, demographics, authors vi cac cot nay thuong se co nhieu gia tri cung loai long nhau
    list_cols = ['genres', 'themes', 'demographics', 'authors']
    for col in list_cols:
        if col in df.columns:
            # Giu list de xu ly Graph, vi cac thuoc tinh cung loai la cac nut rieng biet nen phai noi rieng biet trong do thi
            df[col] = df[col].apply(extract_names)

    # Lay 1 image url duy nhat dai dien cho truyen, xoa bot cac image thua
    def extract_image_url(image_dict):
        try:
            return image_dict.get('jpg', {}).get('large_image_url') or \
                   image_dict.get('jpg', {}).get('image_url')
        except:
            return None

    if 'images' in df.columns:
        df['main_picture'] = df['images'].apply(extract_image_url)
        df.drop(columns=['images'], inplace=True) 

    # Xu ly thoi gian (su dung nam bat dau lam chuan, de tranh thuoc tinh NULL)
    def extract_year(published_dict):
        try:
            return published_dict.get('prop', {}).get('from', {}).get('year')
        except:
            return None
            
    if 'published' in df.columns:
        df['start_year'] = df['published'].apply(extract_year)
        df.drop(columns=['published'], inplace=True)

    # Xu ly thuoc tinh bi rong
    # RAG nhan synopsis lam text chinh de gioi thieu truyen
    # Neu co background thi chep them background vao synopsis
    # Neu khong co thi dien mac dinh
    # Xu ly xong drop cot background de giam kich co file

    # Khoi tao rong cho cac vung bi NaN de cong chuoi
    if 'synopsis' in df.columns:
        df['synopsis'] = df['synopsis'].fillna('')
    else:
        df['synopsis'] = ''
        
    if 'background' in df.columns:
        df['background'] = df['background'].fillna('')
    else:
        df['background'] = ''

    # Gop background vao synopsis
    def combine_synopsis_background(row):
        synopsis = str(row['synopsis']).strip()
        background = str(row['background']).strip()
        content = []
        
        if synopsis and synopsis != 'nan':
            content.append(synopsis)
            
        if background and background != 'nan':
            # Bo sung them background vao synopsis
            content.append(f"\n[Background Info]: {background}")
            
        if not content:
            return "No synopsis available."
            
        return "\n".join(content)

    # Ap dung vao df
    df['synopsis'] = df.apply(combine_synopsis_background, axis=1)
    
    # Xoa cot background sau khi gop xong
    if 'background' in df.columns:
        df.drop(columns=['background'], inplace=True)
    
    # Fill 0 cho cac thuoc tinh numeric bi thieu du lieu
    fill_zero_cols = ['score', 'members', 'favorites', 'chapters', 'volumes']
    for col in fill_zero_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Luu file json
    print(f"Bo du lieu sau khi rut gon: {df.shape[0]} dong, {df.shape[1]} cot.")
    
    # Xuat file json moi
    df.to_json(output_file, orient='records', force_ascii=False, indent=4)
    print(f" File json moi duoc luu tai: {output_file}")


clean_and_drop_attributes('data/raw/manga_data.json', 'manga_data_cleaned.json')

Bo du lieu co 81539 dong, 30 cot
Bo du lieu sau khi rut gon: 81539 dong, 21 cot.
 File json moi duoc luu tai: manga_data.json
